# 03.2 - Probability Distributions

**Phase:** 03 - Statistics & Probability
**Status:** VERIFIED
---

## What Are We Solving?
A probability distribution describes how likely different outcomes are. It is a model of the data-generating process.

## Mental Model
A distribution is a recipe for generating data. The parameters control the recipe.

## Core Concepts
- discrete: Bernoulli, Binomial, Poisson, Categorical
- continuous: Normal, Uniform, Exponential, Beta, Gamma
- parameters: location, scale, shape
- sampling from distributions
- probability density/mass functions
- cumulative distribution functions

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Discrete distributions
print("=== DISCRETE DISTRIBUTIONS ===")

# Bernoulli: single trial, success/failure
p = 0.3
bernoulli_samples = np.random.binomial(n=1, p=p, size=10000)
print(f"Bernoulli(p={p}): mean={bernoulli_samples.mean():.3f}, var={bernoulli_samples.var():.3f}")
print(f"  Theoretical: mean={p:.3f}, var={p*(1-p):.3f}")

# Binomial: n trials, count of successes
n, p = 10, 0.3
binomial_samples = np.random.binomial(n=n, p=p, size=10000)
print(f"\nBinomial(n={n}, p={p}): mean={binomial_samples.mean():.3f}, var={binomial_samples.var():.3f}")
print(f"  Theoretical: mean={n*p:.3f}, var={n*p*(1-p):.3f}")

# Poisson: count of events in interval
lam = 3
poisson_samples = np.random.poisson(lam=lam, size=10000)
print(f"\nPoisson(λ={lam}): mean={poisson_samples.mean():.3f}, var={poisson_samples.var():.3f}")
print(f"  Theoretical: mean={lam:.3f}, var={lam:.3f}")

=== DISCRETE DISTRIBUTIONS ===
Bernoulli(p=0.3): mean=0.289, var=0.205
  Theoretical: mean=0.300, var=0.210

Binomial(n=10, p=0.3): mean=3.028, var=2.139
  Theoretical: mean=3.000, var=2.100

Poisson(λ=3): mean=2.999, var=2.989
  Theoretical: mean=3.000, var=3.000


In [2]:
# Continuous distributions
print("=== CONTINUOUS DISTRIBUTIONS ===")

# Normal: symmetric, bell-shaped
normal_samples = np.random.normal(loc=0, scale=1, size=10000)
print(f"Normal(μ=0, σ=1): mean={normal_samples.mean():.3f}, std={normal_samples.std():.3f}")

# Uniform: equal probability in range
uniform_samples = np.random.uniform(low=0, high=1, size=10000)
print(f"Uniform(0, 1): mean={uniform_samples.mean():.3f}, std={uniform_samples.std():.3f}")

# Exponential: time between events
exp_samples = np.random.exponential(scale=2, size=10000)
print(f"Exponential(scale=2): mean={exp_samples.mean():.3f}, std={exp_samples.std():.3f}")

# Beta: proportions, bounded [0,1]
beta_samples = np.random.beta(a=2, b=5, size=10000)
print(f"Beta(2, 5): mean={beta_samples.mean():.3f}, std={beta_samples.std():.3f}")

# Gamma: positive, skewed
gamma_samples = np.random.gamma(shape=2, scale=2, size=10000)
print(f"Gamma(shape=2, scale=2): mean={gamma_samples.mean():.3f}, std={gamma_samples.std():.3f}")

=== CONTINUOUS DISTRIBUTIONS ===
Normal(μ=0, σ=1): mean=-0.004, std=1.014
Uniform(0, 1): mean=0.502, std=0.288
Exponential(scale=2): mean=2.002, std=2.004
Beta(2, 5): mean=0.285, std=0.158
Gamma(shape=2, scale=2): mean=4.004, std=2.826


## Decision Guidance: Choosing a Distribution

| Data Type | Candidate Distributions |
|---|---|
| Binary outcome | Bernoulli |
| Count of successes in n trials | Binomial |
| Count of events in interval | Poisson |
| Continuous, symmetric | Normal |
| Continuous, positive, skewed | Log-normal, Gamma, Exponential |
| Proportions/percentages | Beta |

In [3]:
# Visualize distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

distributions = [
    ("Bernoulli", bernoulli_samples, "discrete"),
    ("Binomial", binomial_samples, "discrete"),
    ("Poisson", poisson_samples, "discrete"),
    ("Normal", normal_samples, "continuous"),
    ("Uniform", uniform_samples, "continuous"),
    ("Exponential", exp_samples, "continuous"),
    ("Beta", beta_samples, "continuous"),
    ("Gamma", gamma_samples, "continuous"),
]

for ax, (name, samples, dtype) in zip(axes, distributions):
    if dtype == "discrete":
        # For discrete, use bar plot of PMF
        values, counts = np.unique(samples, return_counts=True)
        ax.bar(values, counts / len(samples), alpha=0.7, edgecolor='black')
        ax.set_xticks(values)
    else:
        ax.hist(samples, bins=40, density=True, alpha=0.7, edgecolor='black')
        # Overlay theoretical PDF
        x = np.linspace(samples.min(), samples.max(), 100)
        if name == "Normal":
            ax.plot(x, stats.norm.pdf(x, 0, 1), 'r-', lw=2, label='PDF')
        elif name == "Uniform":
            ax.plot(x, stats.uniform.pdf(x, 0, 1), 'r-', lw=2, label='PDF')
        elif name == "Exponential":
            ax.plot(x, stats.expon.pdf(x, scale=2), 'r-', lw=2, label='PDF')
        elif name == "Beta":
            ax.plot(x, stats.beta.pdf(x, 2, 5), 'r-', lw=2, label='PDF')
        elif name == "Gamma":
            ax.plot(x, stats.gamma.pdf(x, 2, scale=2), 'r-', lw=2, label='PDF')
        ax.legend(fontsize=8)
    ax.set_title(name)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density / Probability')

plt.tight_layout()
plt.savefig('probability_distributions.png', dpi=150, bbox_inches='tight')
print("Saved: probability_distributions.png")

Saved: probability_distributions.png


## Fitting Distributions to Data (MLE)
Maximum Likelihood Estimation finds parameters that make observed data most probable.

In [4]:
# Fit Normal distribution to data using MLE
data = np.random.normal(loc=5, scale=2, size=500)

# MLE for Normal: mean = sample mean, std = sample std (with n denominator)
mle_mean = np.mean(data)
mle_std = np.std(data, ddof=0)  # MLE uses n, not n-1

print(f"True params: μ=5, σ=2")
print(f"MLE estimates: μ={mle_mean:.3f}, σ={mle_std:.3f}")

# Fit using scipy (also MLE for Normal)
from scipy.stats import norm
fitted_params = norm.fit(data)
print(f"Scipy fit: μ={fitted_params[0]:.3f}, σ={fitted_params[1]:.3f}")

# Visualize fit
x = np.linspace(data.min(), data.max(), 100)
plt.hist(data, bins=30, density=True, alpha=0.5, label='Data')
plt.plot(x, norm.pdf(x, *fitted_params), 'r-', lw=2, label='Fitted Normal')
plt.legend()
plt.title('MLE Fit: Normal Distribution')
plt.savefig('mle_fit_normal.png', dpi=150, bbox_inches='tight')
print("Saved: mle_fit_normal.png")

True params: μ=5, σ=2
MLE estimates: μ=4.960, σ=1.983
Scipy fit: μ=4.960, σ=1.983


Saved: mle_fit_normal.png


## Common Mistakes
- using Normal for bounded or skewed data
- ignoring parameter constraints (e.g., variance > 0)
- confusing probability mass with density
- assuming independence when data is correlated

## Hands-On Practice
1. **Basic**: Sample from and plot 5 common distributions.
2. **Guided**: Fit a distribution to real data using MLE.
3. **Independent**: Simulate a process (e.g., customer arrivals) with appropriate distribution.
4. **Challenge**: Explain why heights are approximately Normal but incomes are not.

## Knowledge Check
1. What is the difference between a probability mass function and a probability density function?
2. When would you use a Poisson distribution vs a Binomial distribution?
3. Why is the Normal distribution so common? (Central Limit Theorem)
4. What does the Beta distribution model?
5. How do you choose the right distribution for your data?

In [5]:
# Verification
print("VERIFICATION PASSED: Phase 03.2 complete")
print("Key takeaway: Choose distributions based on data type and generating process.")

VERIFICATION PASSED: Phase 03.2 complete
Key takeaway: Choose distributions based on data type and generating process.


## Summary
- Discrete: Bernoulli (binary), Binomial (counts), Poisson (rates)
- Continuous: Normal (symmetric), Exponential (wait times), Beta (proportions), Gamma (positive skewed)
- Parameters: location (center), scale (spread), shape (skewness)
- MLE: find parameters maximizing likelihood of observed data
- Always visualize fitted distribution against data

## Further Experiment
- Fit multiple distributions to the same data and compare AIC/BIC
- Simulate a queueing system with Poisson arrivals and Exponential service times
- Explore mixture models (e.g., Gaussian mixture for bimodal data)
- Use `scipy.stats.fit()` for automated distribution fitting

## Verification Status
- **STATUS: VERIFIED**
- **EXECUTION: PASS**
- **DEPENDENCIES:** numpy, matplotlib, scipy
- **OUTPUTS: PASS**
- **LAST VERIFIED: 2026-08-29**